# Creating AI without such libraries as Torch or TensorFlow

## Load dataset and STOI

In [14]:
import numpy as np
import json

data = np.memmap("train.bin", dtype=np.uint32, mode="r")

with open("stoi.json", "r") as f:
    stoi = json.load(f)

print(f"Vocabulary length: {len(stoi)}")
print(f"Tokens length: {len(data)}")

FileNotFoundError: [Errno 2] No such file or directory: 'train.bin'

## Split dataset into training and test

In [ ]:
data = data

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [ ]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else test_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

# Streaming batching as there is too much data

In [ ]:
import numpy as np
from tokenizers import Tokenizer
import random

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

file_path = "train.txt"

def sample_story():
    with open(file_path, "r", encoding="utf-8") as f:

        file_size = f.seek(0, 2)
        pos = random.randint(0, file_size - 10000)

        f.seek(pos)

        # skip partial line
        f.readline()

        chunk = f.read(20_000)

    parts = chunk.split("endoftext")

    if len(parts) < 2:
        return sample_story()

    return random.choice(parts).strip()

In [ ]:
def get_batch(block_size, batch_size):

    x_batch = []
    y_batch = []

    while len(x_batch) < batch_size:

        story = sample_story()

        if not story:
            continue

        tokens = tokenizer.encode(story).ids

        if len(tokens) <= block_size + 1:
            continue

        start = np.random.randint(0, len(tokens) - block_size - 1)

        x = tokens[start:start + block_size]
        y = tokens[start + 1:start + block_size + 1]

        x_batch.append(x)
        y_batch.append(y)

    return np.array(x_batch), np.array(y_batch)

## Traing loop

In [15]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        
        logits, _ = model.forward(idx_cond, np.array([1]))
        
        logits = logits[:, -1, :]
        
        max_logits = np.max(logits, axis=-1, keepdims=True)
        exp_logits = np.exp(logits - max_logits)
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        next_tokens = []
        for b in range(probs.shape[0]):
            next_token = np.random.choice(probs.shape[-1], p=probs[b])
            next_tokens.append(next_token)
        
        next_token = np.array(next_tokens).reshape(-1, 1)
        
        idx = np.concatenate([idx, next_token], axis=1)
    
    return idx

In [16]:
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

def check_model_output(model):
    prompt = "History "

    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = generate(model, context, 30)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

In [ ]:
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 128
vocabulary_size = 32_000
block_size = 64
batch_size = 32
block_layers = 4
gradient = Adam(lr=3e-3, warmup_steps=1000, min_lr=1e-5)

model = MiniGPT(vocabulary_size, d_model, block_size, block_layers, gradient)
# model = MiniGPT.__new__(MiniGPT)
# model = model.load("saved_model")

ema_loss = None
for step in range(10_000):
    xb, yb = get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step % 100 == 0:
        check_model_output(model)
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        model.save("saved_model")

model.save("saved_model")


History Tini conscious icebergs Smokey pastime uar fluttered ately offered wettest Adelina curd hids grown daffodils wrrr measures Mount Airport Hello cellent Boppy Abbie scuba Per Occasionally turtle Noon Laws flipped
step 0, lr 0.000010, loss 10.4232, ema_loss 10.4232


In [ ]:
from NoTorchAI.LLM.MiniGPT import MiniGPT


# model = MiniGPT.__new__(MiniGPT)
# model: MiniGPT = model.load("saved_model")

itos = {i: ch for ch, i in stoi.items()}

prompt = "History "
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.uint32)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 30)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

History Waldych bodyscott Hoch Damhoca ase ballasts Stunder ap yrtactier Rite Bedum em legisco Faise ani hemrate Morhuish viv biotek signalistically Meisturb ant Taurani Noku Nito reverently predications Coulaurin acboomero Mov
